In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

from QuantNado import BamStore
from QuantNado.downstream import (
    extract_metadata,
    feature_counts,
    load_gtf,
    plot_pca_scatter,
    plot_pca_scree,
    reduce_byranges_signal,
    run_pca,
)


In [ ]:
fig_dir = Path("./figures")
fig_dir.mkdir(exist_ok=True)

# Load Dataset

In [ ]:
ds = BamStore.open("dataset_full", backend="zarr")
ds

# Explore Zarr Dataset



In [ ]:
print(ds.attrs["sample_names"])
print(ds.attrs["assay_by_sample"])

In [ ]:
# Check dimensions and coordinates
print("Dimensions:", ds.dims)
print("\nCoordinates:", list(ds.coords))
print("\nAttributes:", ds.attrs)
print("\nData variables:", list(ds.data_vars))

In [ ]:
# Helpers for ragged coordinates and metadata
metadata_df = extract_metadata(ds)

# Reduce by BED file

In [ ]:
promoters = "/Users/catherine/work/project/QuantNado/data/hg38/promoters_1024bp.bed"

promoter_ds = reduce_byranges_signal(ds["signal"], bed_file=promoters, reduction="mean")

In [ ]:
# Subset promoter signal to RNA samples
assay_by_sample = list(ds.attrs.get("assay_by_sample", []))
sample_names = [str(s) for s in ds.attrs.get("sample_names", [])]
rna_samples = [s for s, a in zip(sample_names, assay_by_sample) if a == "RNA"]

sample_dim = "sample" if "sample" in promoter_ds.dims else "sample_id"
promoter_ds = promoter_ds.assign_coords({sample_dim: sample_names})

coord_samples = [str(s) for s in promoter_ds.coords[sample_dim].values.tolist()]
valid_samples = [s for s in rna_samples if s in coord_samples]

promoter_rna_ds = promoter_ds.sel({sample_dim: valid_samples})
print("RNA samples in promoter_rna_ds:", promoter_rna_ds.coords[sample_dim].values)


## PCA analysis

Perform Principal Component Analysis on the promoter dataset to visualize sample relationships:

In [ ]:
n_features = 200
rng = np.random.default_rng(42)

promoter_signal32 = promoter_rna_ds["mean"].astype("float32")
feature_count = promoter_signal32.sizes.get("ranges", 0)
keep = min(n_features, feature_count)
subset_idx = rng.choice(feature_count, size=keep, replace=False)
subset_signal = promoter_signal32.isel(ranges=subset_idx).chunk({"ranges": 2000})

In [ ]:
pca_object, pca_result = run_pca(
    input_array=subset_signal,
    n_components=2,
    nan_handling_strategy="drop",
    standardize=True,
    random_state=42,
    subset_size=None,
    subset_strategy=None,
    svd_solver="randomized",
)

In [ ]:
plot_pca_scree(
    pca_object=pca_object,
    filepath=f"{fig_dir}/pca_rna_scree.png",
)



In [ ]:
metadata_rna_df = metadata_df[metadata_df["assay"] == "RNA"]

plot_pca_scatter(
    pca_object,
    pca_result,
    xaxis_pc=1,
    yaxis_pc=2,
    metadata_df=metadata_rna_df,
    colour_by="treatment",
    shape_by="assay",
    filepath=f"{fig_dir}/pca_rna_plot.png",
)

# Extract Feature counts

In [ ]:
gtf_file = "/Users/catherine/work/project/QuantNado/data/hg38/hg38.ncbiRefSeq.gtf"
gtf = load_gtf(gtf_file)
print(gtf.head())

In [ ]:
counts_transcript, feature_meta = feature_counts(
    dataset=ds,
    gtf_file=gtf_file,
    feature_type="transcript",
    assay="RNA",
    integerize=True,
    filter_zero=True,
)
display(feature_meta.head())

In [ ]:
# Align RNA counts and metadata
metadata_rna = metadata_df.loc[metadata_df["assay"] == "RNA"].copy()
metadata_rna.index = metadata_rna["sample"].astype(str)
display(metadata_rna)

## Perform DeSeq

In [ ]:
# Collapse duplicated feature labels to avoid DESeq2 reindex errors
counts_transcript = counts_transcript.groupby(level=0).sum()
feature_meta = feature_meta.groupby(feature_meta.index).first()

counts_df = counts_transcript.T

dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata_rna,
    design="~treatment",
)

dds.fit_size_factors()
dds.obs["size_factors"]

dds.fit_genewise_dispersions()
dds.var["genewise_dispersions"]

dds.fit_dispersion_trend()
dds.uns["trend_coeffs"]
dds.var["fitted_dispersions"]

dds.fit_dispersion_prior()
print(
    f"logres_prior={dds.uns['_squared_logres']}, sigma_prior={dds.uns['prior_disp_var']}"
)

dds.fit_MAP_dispersions()
dds.var["MAP_dispersions"]
dds.var["dispersions"]

dds.fit_LFC()
dds.varm["LFC"]

dds.calculate_cooks()
if dds.refit_cooks:
    dds.refit()

In [ ]:
deseq = DeseqStats(
    dds,
    contrast=np.array([0, 1]),
    alpha=0.05,
    cooks_filter=True,
    independent_filter=True,
)

deseq.run_wald_test()
deseq.p_values

if deseq.independent_filter:
    deseq._independent_filtering()
else:
    deseq._p_value_adjustment()

deseq.padj

deseq.summary()

# Use the coefficient name actually present in results
print("Available LFC coeffs:", deseq.LFC.columns.tolist())
res_shrunk = deseq.lfc_shrink(coeff="treatment[T.MENi]")
if res_shrunk is None:
    print("lfc_shrink returned None; falling back to wald results.")
    res_shrunk = deseq.results_df.copy()
# Keep shrunk (or fallback) results for downstream plotting
res_shrunk = res_shrunk.dropna(subset=["log2FoldChange", "padj"])
print("Shrunk results shape:", res_shrunk.shape)


In [ ]:
# Diagnostics: sample alignment, library sizes, zero genes, and top LFC
print("Counts rows (samples) vs metadata rows:", counts_df.shape[0], metadata_rna.shape[0])
print("Counts columns (genes):", counts_df.shape[1])
print("Metadata index head:", metadata_rna.index.tolist()[:10])
print("Counts index head:", counts_df.index.tolist()[:10])
print("Treatment counts:\n", metadata_rna["treatment"].value_counts())

lib_sizes = counts_df.sum(axis=1)
print("Library sizes summary:\n", lib_sizes.describe())
print("All-zero samples:", (lib_sizes == 0).sum())

# genes completely zero
zero_genes = (counts_df == 0).all(axis=0)
print("Genes all zero:", zero_genes.sum())

# top 10 absolute LFC from results
if "results_df" in dir(deseq):
    res = deseq.results_df.copy()
    res = res.dropna(subset=["log2FoldChange", "padj"]).sort_values("log2FoldChange", key=np.abs, ascending=False)
    print(res[["log2FoldChange", "padj"]].head(10))
else:
    print("Run deseq first to see results.")


In [ ]:
# Inspect top-|LFC| genes: raw and normalized counts
res = deseq.results_df.copy()
res = res.dropna(subset=["log2FoldChange", "padj"]).sort_values(
    "log2FoldChange", key=np.abs, ascending=False
)
top_genes = res.index[:5].tolist()
print("Top-|LFC| genes:", top_genes)
print("Size factors:\n", dds.obs["size_factors"])

print("Raw counts (top genes):")
display(counts_df[top_genes].T)

print("Normalized counts (counts / size_factors):")
norm_counts = counts_df[top_genes].div(dds.obs["size_factors"], axis=0)
display(norm_counts.T)

## Volcano plot

In [ ]:
# Volcano plot from DESeq2 results
# Assumes `res_shrunk` from the DESeq step (shrunk LFCs).
if "res_shrunk" not in globals():
    res = deseq.results_df.copy()
else:
    res = res_shrunk.copy()

res = res.dropna(subset=["padj", "log2FoldChange"])
res["neg_log10_padj"] = -np.log10(res["padj"] + 1e-300)

sig_mask = (res["padj"] < 0.05) & (res["log2FoldChange"].abs() > 1)
up_reg = sig_mask & (res["log2FoldChange"] > 0)
down_reg = sig_mask & (res["log2FoldChange"] < 0)

fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
ax.scatter(
    res["log2FoldChange"],
    res["neg_log10_padj"],
    s=8,
    c="lightgray",
    alpha=0.6,
    label="all genes",
)
ax.scatter(
    res.loc[up_reg, "log2FoldChange"],
    res.loc[up_reg, "neg_log10_padj"],
    s=10,
    c="crimson",
    alpha=0.8,
    label="up (padj<0.05 & |LFC|>1)",
)
ax.scatter(
    res.loc[down_reg, "log2FoldChange"],
    res.loc[down_reg, "neg_log10_padj"],
    s=10,
    c="steelblue",
    alpha=0.8,
    label="down (padj<0.05 & |LFC|>1)",
)
ax.axvline(0, color="k", linestyle="--", linewidth=1)
ax.set_xlabel("log2 fold change (MENi vs DMSO)")
ax.set_ylabel("-log10 adjusted p-value")
ax.legend(fontsize=8, loc="upper right", bbox_to_anchor=(0.7, -0.2))
plt.tight_layout()

plt.show()

# Save figure
fig_path = fig_dir / "volcano_rna_treatment.png"
fig.savefig(fig_path, dpi=150)
print(f"Saved volcano plot to {fig_path}")
